<div>
<img src=https://www.institutedata.com/wp-content/uploads/2019/10/iod_h_tp_primary_c.svg width="300">
</div>

## Lab 4.2.1: Feature Selection

In this lab, we delve into the fundamental concept of feature selection. We start by conducting correlation analysis to identify relevant features for our regression model. By examining the relationship between each feature and the target variable, we aim to pick the most influential features. Additionally, we explore the significance of cross validation in model evaluation and how it relates to feature selection. Through cross validation, we ensure that our model generalises well to unseen data by assessing its performance across multiple validation sets.

### 1. Load & Explore Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

#### 1.1 Load Data

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving winequality_merged.csv to winequality_merged.csv


In [ ]:
# Read CSV
wine_csv = 'winequality_merged.csv'
df=pd.read_csv(wine_csv)

#### 1.2 Explore Data (Exploratory Data Analysis)

In [ ]:
# ANSWER

print(df.head())

print(df.info())

# Summary statistics
print(df.describe())


   fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0            7.4              0.70         0.00             1.9      0.076   
1            7.8              0.88         0.00             2.6      0.098   
2            7.8              0.76         0.04             2.3      0.092   
3           11.2              0.28         0.56             1.9      0.075   
4            7.4              0.70         0.00             1.9      0.076   

   free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
0                 11.0                  34.0   0.9978  3.51       0.56   
1                 25.0                  67.0   0.9968  3.20       0.68   
2                 15.0                  54.0   0.9970  3.26       0.65   
3                 17.0                  60.0   0.9980  3.16       0.58   
4                 11.0                  34.0   0.9978  3.51       0.56   

   alcohol  quality  red_wine  
0      9.4        5         1  
1      9.8        5   

In [ ]:
# Check for missing values
# Total missing values per column
print(df.isnull().sum())


fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
red_wine                0
dtype: int64


In [ ]:
# Check for duplicates

print("Number of duplicate rows:", df.duplicated().sum())


Number of duplicate rows: 1177


### 2. Set Target Variable

Create a target variable for wine quality.

In [ ]:
# Target Variable = quality
df['quality_label'] = np.where(df['quality'] >= 7, 1, 0)


### 3. Set Predictor Variables

Create a predictor matrix with variables of your choice. State your reasoning for the choices you make.

In [14]:
# ANSWER
predictor_columns = [
    'volatile acidity', # High levels can cause a vinegar taste
    'citric acid', # Adds freshness and acidity
    'residual sugar', # Affects sweetness
    'chlorides', # High concentrations (like saltiness) can negatively impact flavor.
    'free sulfur dioxide', # may affect freshness and aroma preservation.
    'density', # Related to alcohol and sugar content
    'pH', # Influences stability and freshness
    'sulphates', # Enhance shelf life
    'alcohol' # Strong influence on taste and body
]

X = df[predictor_columns]
y = df['quality_label']

### 4. Using Linear Regression Create a Model and Test Score

In [15]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

In [16]:
# Train-Test Split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [17]:
# Create a model for Linear Regression
model = LinearRegression()

# Fit the model with the Training data
model.fit(X_train, y_train)

# Calculate the score (R^2 for Regression) for Training Data
train_score = model.score(X_train, y_train)
print("Training R^2 Score:", train_score)

# Calculate the score (R^2 for Regression) for Testing Data
test_score = model.score(X_test, y_test)
print("Testing R^2 Score:", test_score)

Training R^2 Score: 0.1813849507311629
Testing R^2 Score: 0.1867102924425078


## BONUS: Cross validation

In [21]:
# Cross validation
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score


In [22]:
# Set up 5-fold cross validation
k_fold = KFold(5, shuffle=True)
train_scores = []
train_rmse = []
test_scores = []
test_rmse = []

for k, (train, test) in enumerate(k_fold.split(X)):

    # Get training and test sets for X and y
    X_train, X_test = X.iloc[train], X.iloc[test]
    y_train, y_test = y.iloc[train], y.iloc[test]
    # Fit model with training set
    model = LinearRegression()
    model.fit(X_train, y_train)
    # Make predictions with training and test set
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    # Score R2 and RMSE on training and test sets and store in list
    r2_train = r2_score(y_train, y_train_pred)
    r2_test = r2_score(y_test, y_test_pred)
    train_scores.append(r2_train)
    test_scores.append(r2_test)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
    rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
    train_rmse.append(rmse_train)
    test_rmse.append(rmse_test)
# Create a metrics_df dataframe to display r2 and rmse scores
metrics_df = pd.DataFrame({
    'Fold': range(1, 6),
    'Train R2': train_scores,
    'Train RMSE': train_rmse,
    'Test R2': test_scores,
    'Test RMSE': test_rmse
})

print(metrics_df)
print("\nAverage Test R2:", np.mean(test_scores))
print("Average Test RMSE:", np.mean(test_rmse))

   Fold  Train R2  Train RMSE   Test R2  Test RMSE
0     1  0.176218    0.360884  0.205729   0.353365
1     2  0.186095    0.358714  0.167152   0.361845
2     3  0.187819    0.358176  0.160294   0.363978
3     4  0.183276    0.358244  0.178879   0.363609
4     5  0.180404    0.360205  0.190803   0.355706

Average Test R2: 0.1805716551604745
Average Test RMSE: 0.3597004961564032


In [ ]:
# Describe the metrics
# R^2 tells you how well your model explains the variance in the target variable. Higher R^2 means the model explains more of the variation in wine quality.
# RMSE is the average prediction error, in the same units as the target. Lower RMSE = better performance.

### 5. Feature Selection

What's your score (R^2 for Regression) for Testing Data?

0.1806

How many feature have you selected? Can you improve your score by selecting different features?

I selected 9 features. I could improve the score by selecting more correlated features.



**Please continue with Lab 4.2.2 with the same dataset.**



---



---



> > > > > > > > > © 2025 Institute of Data


---



---



